# Regulatory and marker evidence for candidate Muraro cell reannotation

This notebook reproduces the logic used to investigate a small group of cells originally annotated as ductal. It combines expression-space visualization, known marker genes and scDGRN-derived TF regulatory activity. The analysis supports a candidate reannotation; it does not treat computational labels as definitive cell identity.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the scDGRN repository.")


REPO_ROOT = find_repository_root()
sys.path.insert(0, str(REPO_ROOT))

DATASET_NAME = "muraro"
RESULT_NAME = os.environ.get("SCDGRN_RESULT_NAME", "muraro")
DATASETS_ROOT = Path(os.environ.get("SCDGRN_DATASETS_ROOT", REPO_ROOT / "datasets")).resolve()
RESULTS_ROOT = Path(os.environ.get("SCDGRN_RESULTS_ROOT", REPO_ROOT / "results")).resolve()
DATASET_DIR = DATASETS_ROOT / DATASET_NAME
RESULT_DIR = RESULTS_ROOT / RESULT_NAME
OUTPUT_DIR = RESULT_DIR / "tutorial_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMN = "cell_type"
RANDOM_SEED = 3407
TOP_N_EDGES = 50

print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATASET_DIR}")
print(f"Results:    {RESULT_DIR}")
print(f"Outputs:    {OUTPUT_DIR}")


## 1. Load the candidate cells and expression matrix


In [ ]:
import scanpy as sc

candidate_cells = [
    "D30.7_88", "D28.2_79", "D31.7_43", "D30.5_41", "D29.2_49",
    "D30.2_38", "D31.6_57", "D31.5_72", "D28.4_1", "D30.2_79",
    "D30.3_84", "D28.3_88", "D28.3_17", "D28.2_66", "D30.8_21",
]

adata = sc.read_csv(DATASET_DIR / "ExpressionData.csv")
labels = pd.read_csv(DATASET_DIR / "cell_data.csv")[LABEL_COLUMN].astype(str)
adata.obs[LABEL_COLUMN] = labels.to_numpy()
adata.obs["annotation_check"] = labels.to_numpy()
candidate_mask = adata.obs_names.astype(str).isin(candidate_cells)
adata.obs.loc[candidate_mask, "annotation_check"] = "candidate myeloid"

print(f"Matched {candidate_mask.sum()} of {len(candidate_cells)} candidate cells.")
if candidate_mask.sum() != len(candidate_cells):
    missing = sorted(set(candidate_cells) - set(adata.obs_names.astype(str)))
    print("Missing cell identifiers:", missing)


## 2. Visualize the candidate population


In [ ]:
sc.pp.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=min(40, adata.obsm["X_pca"].shape[1]))
sc.tl.umap(adata, random_state=RANDOM_SEED)

sc.pl.umap(
    adata,
    color=[LABEL_COLUMN, "annotation_check"],
    frameon=False,
    show=False,
)
plt.savefig(OUTPUT_DIR / "muraro_candidate_myeloid_umap.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "muraro_candidate_myeloid_umap.svg", bbox_inches="tight")
plt.show()


## 3. Compare established myeloid and ductal marker genes


In [ ]:
marker_genes = {
    "myeloid": ["ITGAM", "CD33", "CSF1R", "CD68", "CD163", "C1QA"],
    "ductal": ["MMP7", "KRT19", "CFTR", "TSPAN8", "HNF1B"],
}
marker_genes = {
    group: [gene for gene in genes if gene in adata.var_names]
    for group, genes in marker_genes.items()
}

sc.pl.dotplot(
    adata,
    marker_genes,
    groupby="annotation_check",
    standard_scale="var",
    cmap="YlOrRd",
    show=False,
)
plt.savefig(OUTPUT_DIR / "muraro_candidate_marker_dotplot.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "muraro_candidate_marker_dotplot.svg", bbox_inches="tight")
plt.show()


## 4. Compare inferred TF regulatory activity


In [ ]:
from tutorial.tutorial_utils import compute_tf_activity

tf_names = pd.read_csv(DATASET_DIR / "TF.csv", header=None).iloc[:, 0].astype(str).to_numpy()
activity_path = OUTPUT_DIR / "muraro_tf_activity.npy"
if activity_path.exists():
    tf_activity = np.load(activity_path)
else:
    tf_activity = compute_tf_activity(RESULT_DIR / "single_network", adata.n_obs, len(tf_names))
    np.save(activity_path, tf_activity)

activity = pd.DataFrame(tf_activity, columns=tf_names, index=adata.obs_names)
candidate_mean = activity.loc[candidate_mask].mean()
ductal_mean = activity.loc[labels.eq("ductal").to_numpy() & ~candidate_mask].mean()
comparison = pd.DataFrame({
    "candidate_mean": candidate_mean,
    "ductal_mean": ductal_mean,
    "difference": candidate_mean - ductal_mean,
}).sort_values("difference", ascending=False)
comparison.to_csv(OUTPUT_DIR / "muraro_candidate_vs_ductal_tf_activity.csv")
comparison.head(20)


## Interpretation boundary

Concordant myeloid markers and regulatory signatures support re-examining the original ductal annotation. Definitive reassignment should additionally consider the original raw counts, quality-control information and independent biological validation.
